# AI weather forecasting in one hour: Storm Boris, September 2024

**CAS in Machine Learning, ETH Zurich**

Storm Boris, stalled over Central Europe on 12-16 September 2024 and dropped
record rainfall on Czechia, Austria and Poland. We forecast it with an AI weather model called **Pangu-Weather** and downscale ERA5 to kilometre scale. The whole session runs on a free Colab **T4 GPU**.

| # | Section | Time |
|---|---|---|
| 0 | Runtime + install `earth2studio` | 10 min |
| 1 | Pull ERA5 for the case from Google's ARCO archive | 5 min |
| 2 | **Pangu-Weather** daily forecast on the GPU | 15 min |
| 3 | Verify against ERA5, persistence, climatology | 10 min |
| 4 | Downscale to **2.2 km hourly** with CorrDiff, plus a small ensemble | 20 min |

### Takeaways

1. A global AI model is a few hundred MB of weights and one loop over autoregressive steps. The
   work is the data plumbing and the verification, not the model call.
2. Skill is hard to interpret without a baseline. Compute persistence and climatology first.
3. Pangu is *deterministic*: one initial condition, one forecast, no free ensemble. CorrDiff's diffusion sampler provides a small ensemble of forecasts, which is useful for uncertainty quantification.
4. Global models run at ~28 km, and Pangu carries no precipitation at all -- too
   coarse and too incomplete for Alpine rain. Generative downscaling fixes both: 2.2 km, hourly,
   with rain.

> **Runtime**: `Runtime > Change runtime type > T4 GPU`, before running anything.

---
## 0. Runtime and installation

This is the **worksheet** version: every cell with a `# TODO` is for you to fill in, and the comment just above it is the hint. A fully worked copy lives in the course repo under `.instructor/`.

Check the hardware.

In [ ]:
import subprocess
import sys

print("Python", sys.version.split()[0])

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
HAS_GPU = gpu.returncode == 0
if HAS_GPU:
    print("GPU:", gpu.stdout.strip())
else:
    print(
        "No GPU visible. The model cells (Pangu, CorrDiff) will replay their\n"
        "cached_outputs/ instead of running; everything else is unchanged.\n"
        "On Colab: Runtime > Change runtime type > T4 GPU, then re-run."
    )

The install, briefly:

- **Pangu-Weather** ships as an ONNX graph and runs through `onnxruntime-gpu`
- **CorrDiff** needs `nvidia-physicsnemo` and `natten` (neighbourhood attention, what makes km-scale attention affordable).

Takes 2-4 minutes.

In [ ]:
%pip install -q uv

import os
import pathlib
import re
import subprocess
import sys
import time
import urllib.request

try:
    import torch
except ModuleNotFoundError:
    # No-GPU machine: a CPU torch build is enough to replay the cache.
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch",
            "--index-url",
            "https://download.pytorch.org/whl/cpu",
        ],
        check=True,
    )
    import torch

print(f"python {sys.version.split()[0]}   torch {torch.__version__}   cuda {torch.version.cuda}")

# Multi-GB model downloads; raise the timeout so a nearly-done download doesn't abort.
os.environ["EARTH2STUDIO_PACKAGE_TIMEOUT"] = "1800"

# Pin the torch already present so nothing swaps it.
pathlib.Path("constraints.txt").write_text(f"torch=={torch.__version__.split('+')[0]}\n")


def pip(*args, **env):
    """uv pip install that raises on failure, unlike a bare ! magic."""
    subprocess.run(
        ["uv", "pip", "install", "--system", "-q", "--constraint", "constraints.txt", *args],
        check=True,
        env={**os.environ, **env},
    )

In [ ]:
try:
    import earth2studio  # already provisioned (e.g. the CSCS GH200 container)

    print("earth2studio already present -- skipping install")
except ModuleNotFoundError:
    if not HAS_GPU:
        # No GPU: model cells replay the cache, so onnxruntime-gpu / NATTEN /
        # physicsnemo are not needed. earth2studio still does the ERA5 fetches.
        t0 = time.perf_counter()
        pip("earth2studio==0.17.0", "cartopy")
        print(f"lightweight (no-GPU) install in {time.perf_counter() - t0:.0f} s")
    else:
        t0 = time.perf_counter()
        pip(
            "earth2studio==0.17.0",
            # physicsnemo from the git rev earth2studio pins for the cosmo extra, not PyPI 2.2.0:
            "nvidia-physicsnemo @ git+https://github.com/NVIDIA/physicsnemo.git"
            "@ced75d93d014f70bb691372788eee2d201171c12",
            "einops>=0.8.1",
            "nvtx",
            "cartopy",
            # onnxruntime for Pangu. Must be in place before earth2studio's first import
            "onnxruntime-gpu==1.26.0",
            "onnx==1.21.0",
        )
        print(f"everything else installed in {time.perf_counter() - t0:.0f} s")

        # NATTEN for the CorrDiff downscaler, also needed before earth2studio's first import
        _tv = torch.__version__.split("+")[0].split(".")
        TORCH_TAG = f"torch{_tv[0]}{_tv[1]}0"
        CUDA_TAG = "cu" + torch.version.cuda.replace(".", "")
        CP = f"cp{sys.version_info.major}{sys.version_info.minor}"

        index = urllib.request.urlopen("https://whl.natten.org/", timeout=60).read().decode()
        matches = re.findall(
            rf'href="([^"]*natten-([\d.]+)%2B{TORCH_TAG}{CUDA_TAG}-{CP}-{CP}-linux_x86_64\.whl)"',
            index,
        )
        if not matches:
            raise RuntimeError(
                f"No NATTEN wheel for {TORCH_TAG}/{CUDA_TAG}/{CP}. Check "
                "https://whl.natten.org/ -- see the README for the fallback."
            )
        NATTEN_URL, NATTEN_VERSION = max(matches, key=lambda m: [int(x) for x in m[1].split(".")])
        pip(NATTEN_URL)
        print(f"natten {NATTEN_VERSION} installed for {TORCH_TAG} {CUDA_TAG} {CP}")

In [ ]:
import gc
import warnings
from collections import OrderedDict

import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr

from IPython.display import HTML
from earth2studio import run
from earth2studio.data import ARCO, WB2Climatology, fetch_data
from earth2studio.io import XarrayBackend
from earth2studio.models.px import Pangu24
from earth2studio.utils.coords import map_coords

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False

print("earth2studio ready, device =", DEVICE, "| cartopy =", HAS_CARTOPY)

---
## 1. The case: Storm Boris

Boris is a cyclone that runs from the Gulf of Genoa north-east around
the Alps. It is a dangerous rain pattern in Central Europe: it wraps warm, moist Mediterranean
air onto the northern flank of the Alps and the Bohemian mountains, where the terrain forces it
to rise. The 2002 Elbe and 2013 Danube floods were both events of the same type.

Boris added a **cut-off low**: an upper-level pocket of cold, low-pressure air that has pinched
off from the jet stream, so with no fast flow left to steer it the system stalls in place -- here
for four days. Totals topped 400 mm (or litres of water) in the Czech and Austrian mountains.

Experiment: initialise at **00 UTC on 9 September 2024**, days before the rain, and see whether
the model builds the cut-off low in the right place at the right time.

In [ ]:
# ---- Experiment configuration -------------------------------------------------
QUICK = False  # True -> shorter forecast and smaller diffusion ensemble, for a first test run

INIT = np.datetime64("2024-09-09T00:00:00")  # forecast initialisation
LEAD_HOURS = 144  # 6 days, to the peak of the event
N_SAMPLES = 8  # CorrDiff diffusion ensemble size -- drives the run and the spread map
N_SHOW = 3  # diffusion sample panels to display (spread is taken over all N_SAMPLES)
N_HOURS = 12  # hourly CorrDiff mean frames ending at DOWNSCALE_TIME; the animation length
HOURS_PER_BATCH = 6  # hours fetched+downscaled per chunk; caps peak memory, lower it on a small GPU
SEED = 0  # base seed for the diffusion sampler; member i draws with SEED + i

if QUICK:
    LEAD_HOURS, N_SAMPLES, N_SHOW, N_HOURS = 72, 4, 2, 6

# Kept from the forecast. Pangu predicts 69: geopotential, specific humidity,
# temperature and winds on 13 pressure levels, plus four surface fields.
# No precipitation and no cloud
VARS = ["msl", "z500", "q700", "t2m", "u10m", "v10m"]

# Plotting / verification window, centred on the cut-off low over Central Europe.
LAT_RANGE = (36.0, 60.0)
LON_RANGE = (1.0, 29.0)

# Downscaling target: CorrDiff's finest, 2.2 km.
DOWNSCALE_TIME = np.datetime64("2024-09-15T00:00:00")  # anchor hour, peak of the event
DOWNSCALE_RES = "rea2"  # "rea2" = 2.2 km, "rea6" = 6 km
# The eastern Alpine rim into Slovenia and the Slovak border ranges -- the
# southern half of Boris's flood zone, trimmed north to keep the diffusion
# ensemble affordable. A wide box (aspect ~2.3); the fringe sits in COSMO-REA2's
# extended margin (a one-time out-of-distribution warning on set_domain).
DOWNSCALE_BBOX = dict(lat_min=45.9, lat_max=49.1, lon_min=12.2, lon_max=19.6)
# N_HOURS consecutive hourly ERA5 steps ending at DOWNSCALE_TIME (ARCO is native hourly).
DOWNSCALE_TIMES = DOWNSCALE_TIME - np.arange(N_HOURS - 1, -1, -1) * np.timedelta64(1, "h")
# Cache key for the CorrDiff inputs: they follow the crop box and the resolution.
DOWNSCALE_TAG = (
    f"{DOWNSCALE_RES}_{DOWNSCALE_BBOX['lat_min']}-{DOWNSCALE_BBOX['lat_max']}"
    f"_{DOWNSCALE_BBOX['lon_min']}-{DOWNSCALE_BBOX['lon_max']}"
)

NSTEPS = LEAD_HOURS // 24  # Pangu-Weather (24 h variant) takes 24 h steps
VALID = INIT + np.arange(NSTEPS + 1) * np.timedelta64(24, "h")
# Cache key for the ERA5 / climatology slices below: they follow INIT and LEAD_HOURS,
# so changing either (exercise 1) misses the cache and refetches instead of lying.
RUN_TAG = f"{str(INIT)[:13].replace('-', '').replace('T', '')}_{LEAD_HOURS}h"

print(f"init {INIT}  ->  {VALID[-1]}   ({NSTEPS} daily steps)")

In [ ]:
# ---- Model cells: run on GPU, or replay from cache ------------------------
# Pangu (section 2) and CorrDiff (section 4) need a GPU. With one, the model cells
# run and rewrite cached_outputs/*.nc; without one, they load the last committed
# run so the rest of the notebook still works.
from pathlib import Path

# The Colab badge opens this notebook on its own, without the repo, so the committed
# cached_outputs/ would be missing and the replay path could not run. Clone it.
if not Path("cached_outputs").exists() and Path("/content").is_dir():
    repo = Path("/content/cas-ml-e2s")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/sadamov/cas-ml-e2s.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    print("working directory ->", os.getcwd())

CACHE_DIR = Path("cached_outputs")
# nbconvert runs the kernel in the notebook's own directory, so an instructor render
# of .instructor/*_solutions.ipynb would otherwise start a second, empty cache there.
if Path.cwd().name == ".instructor":
    CACHE_DIR = Path("..") / "cached_outputs"
CACHE_DIR.mkdir(exist_ok=True)

try:
    import certifi

    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
except ImportError:
    pass

if os.environ.get("FORCE_MODELS"):
    RUN_MODELS = True
elif os.environ.get("FORCE_CACHE"):
    RUN_MODELS = False
else:
    RUN_MODELS = torch.cuda.is_available()
    if RUN_MODELS:
        try:
            import onnxruntime  # noqa: F401
            import natten  # noqa: F401
        except ImportError:
            RUN_MODELS = False

print(
    f"RUN_MODELS = {RUN_MODELS}  "
    + (
        "(model cells run on GPU and refresh cached_outputs/)"
        if RUN_MODELS
        else "(model cells replay cached_outputs/*.nc)"
    )
)


def save_output(name, ds):
    """Write an xarray Dataset to cached_outputs/<name>.nc, compressed."""
    p = CACHE_DIR / f"{name}.nc"
    ds.to_netcdf(p, encoding={v: {"zlib": True, "complevel": 4} for v in ds.data_vars})
    print(f"  cached -> {p}  ({p.stat().st_size / 1e6:.1f} MB)")


def load_output(name):
    """Read cached_outputs/<name>.nc, with a clear error if it was never generated."""
    p = CACHE_DIR / f"{name}.nc"
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found. Run this notebook once on a GPU (or with FORCE_MODELS=1) "
            "to generate cached_outputs/, then commit it."
        )
    print(f"  loaded <- {p}")
    return xr.open_dataset(p, decode_timedelta=True)


def cached(name, build):
    """Return cached_outputs/<name>.nc if it is there, else build it and cache it.

    Unlike load_output this never fails: it just falls back to the live fetch. The
    ERA5 and climatology slices go through here so a session never waits on GCS
    twice, and the committed cache means it usually waits zero times."""
    p = CACHE_DIR / f"{name}.nc"
    if p.exists():
        print(f"  loaded <- {p}")
        return xr.open_dataset(p, decode_timedelta=True)
    ds = build()
    save_output(name, ds)
    return ds


def cached_model_input(name, times, ic):
    """ERA5 on the model's own input grid, as (tensor, coords) ready for `alps`.

    ARCO stores lat/lon as a single chunk, so asking for the 29x45 window over the
    Alps downloads the same global field as asking for the whole planet -- 4.2 MB
    per surface variable per hour, 153.7 MB for a pressure-level one. Cropped and
    cached the result is well under a megabyte, so this is the one fetch worth
    keeping in the repo. Built in HOURS_PER_BATCH chunks so a cold run never holds
    the whole global download at once."""
    times = np.array(times, dtype="datetime64[ns]")
    p = CACHE_DIR / f"{name}.nc"
    if p.exists():
        d = xr.open_dataset(p, decode_timedelta=True)
        print(f"  loaded <- {p}")
        x = torch.as_tensor(d["x"].values, device=DEVICE)
        # netCDF round-trips string coords as object arrays; earth2studio wants str.
        coords = OrderedDict(
            (k, d[k].values.astype(str) if d[k].dtype == object else d[k].values)
            for k in d["x"].dims
        )
        return x, coords

    parts, coords = [], None
    for i0 in range(0, len(times), HOURS_PER_BATCH):
        xb, cb = fetch_data(
            source=arco,
            time=times[i0 : i0 + HOURS_PER_BATCH],
            variable=ic["variable"],
            device=DEVICE,
        )
        xb = xb[:, 0]  # drop the singleton lead_time axis; the hourly time axis stays
        cb = OrderedDict((k, v) for k, v in cb.items() if k != "lead_time")
        xb, coords = map_coords(xb, cb, ic)
        parts.append(xb)
        print(f"  fetched {min(i0 + HOURS_PER_BATCH, len(times))}/{len(times)} hours")

    x = torch.cat(parts)
    coords["time"] = times
    save_output(
        name,
        xr.Dataset(
            {"x": (tuple(coords), x.float().cpu().numpy())},
            coords={k: np.asarray(v) for k, v in coords.items()},
        ),
    )
    return x, coords

### Where the data comes from

**ARCO-ERA5** is Google's cloud-optimised copy of ERA5: one public Zarr store on GCS, 1940 to
present, 0.25 degrees, **native hourly** -- so we slice out exactly the fields and times we need,
including the hourly sequence CorrDiff downscales later.

ERA5 has two roles here: the **initial condition** the forecast starts from, and the **truth** we
score it against at every lead time. The `Cache` wrapper below fetches each slice from GCS once
and reuses it.

In [ ]:
arco = ARCO()
climatology = WB2Climatology("1990-2019_6h_1440x721.zarr")


class Cache:
    """Memoise a data source so the same fields are downloaded once, not once per member."""

    def __init__(self, source):
        self.source = source
        self._store = {}

    def __call__(self, time, variable):
        t = np.atleast_1d(np.asarray(time, dtype="datetime64[ns]"))
        v = tuple(str(x) for x in np.atleast_1d(variable))
        key = (t.tobytes(), v)
        if key not in self._store:
            self._store[key] = self.source(t, list(v))
        return self._store[key]


era5 = Cache(arco)


def to_dataset(da, subset=True):
    """earth2studio DataArray -> tidy Dataset on -180..180 longitudes, cropped to Europe."""
    ds = da.to_dataset("variable")
    ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180)).sortby("lon")
    if subset:
        ds = ds.sel(lat=slice(LAT_RANGE[1], LAT_RANGE[0]), lon=slice(*LON_RANGE))
    return ds

In [ ]:
def make_axes(nrows, ncols, width, extent=None, constrained=True, pad=0.6):
    """Grid of PlateCarree map axes framed to one shared `extent`
    ([lon0, lon1, lat0, lat1], default = the Europe window).

    The figure height is the height the aspect-locked map panels need, plus
    `pad` inches of headroom for the suptitle and any colorbar"""
    lon0, lon1, lat0, lat1 = extent if extent is not None else [*LON_RANGE, *LAT_RANGE]
    height = width / ncols * nrows * (lat1 - lat0) / (lon1 - lon0) + pad
    kw = {"subplot_kw": {"projection": ccrs.PlateCarree()}} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(width, height), layout="constrained" if constrained else None, **kw
    )
    if constrained:
        fig.get_layout_engine().set(w_pad=0.08, h_pad=0.08, wspace=0, hspace=0)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        if HAS_CARTOPY:
            ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.6)
            ax.set_extent([lon0, lon1, lat0, lat1], crs=ccrs.PlateCarree())
    return fig, axes


def draw_wind(ax, ds, stride=6, scale=420, **kw):
    """10 m wind as arrows, subsampled so the field stays readable."""
    sub = ds.isel(lat=slice(None, None, stride), lon=slice(None, None, stride))
    ax.quiver(
        sub.lon,
        sub.lat,
        sub.u10m,
        sub.v10m,
        color="0.15",
        scale=scale,
        width=0.003,
        **GEO,
        **kw,
    )


# Passed to every pcolormesh/contour/quiver call so the data is georeferenced when cartopy is on.
GEO = {"transform": ccrs.PlateCarree()} if HAS_CARTOPY else {}

Three snapshots through the storm's life: total column water vapour (`tcwv`) is the moisture
plume, mean sea level pressure (`msl`) is the cyclone. Live GCS download, ~1-2 min.

In [ ]:
SNAPSHOTS = np.array(["2024-09-11T12", "2024-09-13T12", "2024-09-15T00"], dtype="datetime64[ns]")

overview = cached("era5_overview", lambda: to_dataset(era5(SNAPSHOTS, ["msl", "tcwv"])))
overview

In [ ]:
fig, axes = make_axes(1, 3, 16)

for ax, t in zip(axes, SNAPSHOTS):
    snap = overview.sel(time=t)
    m = ax.pcolormesh(
        snap.lon, snap.lat, snap.tcwv, vmin=5, vmax=45, cmap="BrBG", shading="auto", **GEO
    )
    cs = ax.contour(
        snap.lon,
        snap.lat,
        snap.msl / 100,
        levels=np.arange(960, 1044, 4),
        colors="k",
        linewidths=0.6,
        **GEO,
    )
    ax.clabel(cs, cs.levels[::2], fontsize=6, fmt="%d")
    ax.set_title(str(t)[:13].replace("T", " ") + " UTC", fontsize=10)

fig.colorbar(m, ax=axes, shrink=0.8, label="total column water vapour [kg m$^{-2}$]")
fig.suptitle("ERA5: Storm Boris, moisture plume and sea level pressure")
plt.show()

Left to right: on 11 September the moisture is a plume streaming north from the Mediterranean; by
the 13th the low has closed off over Central Europe and is rotating the plume around itself; by
the 15th it has barely moved -- the same air lifted over the same region for days.

---
## 2. Forecasting with Pangu-Weather

**Pangu-Weather** (Bi et al., 2023) was among the first AI models to beat operational NWP on
headline scores. It is built differently from section 4's diffusion model:

- A **3D Earth-Specific Transformer**: attention over a (level, latitude, longitude) grid with an
  explicit height axis, not pressure levels as extra channels.
- **Deterministic**: one initial condition in, one forecast out, no noise to re-seed.
- Ships as a single 1.1 GB **ONNX** file, run through `onnxruntime-gpu`.
- `Pangu24` takes a **24 h** autoregressive step. Cascaded `Pangu6`/`Pangu3` exist for finer
  steps at roughly double the VRAM; `Pangu24` alone is what fits a free T4.

Its 69 variables are geopotential, specific humidity, temperature and winds on 13 pressure levels
plus four surface fields -- it carries moisture, then, but **no precipitation and no clouds**, and
everything is at ~28 km. That is the gap section 4 fills.

In [ ]:
if RUN_MODELS:
    t0 = time.perf_counter()
    model = Pangu24.load_model(Pangu24.load_default_package()).to(DEVICE)
    print(f"loaded in {time.perf_counter() - t0:.0f} s")

    if DEVICE == "cuda":
        free, total = torch.cuda.mem_get_info()
        print(f"VRAM: {(total - free) / 1e9:.1f} GB used / {total / 1e9:.1f} GB total")
else:
    print("no GPU -- skipping Pangu load; the forecast is read from cached_outputs/")

### Running the rollout

Autoregressive as usual: state in, state a step later out, feed back.
`earth2studio.run.deterministic` is that loop plus the data plumbing; `output_coords` subsets what
gets written (a good habit even when, as here, the full output would be small).

One deterministic forecast, six 24 h steps, no re-seeding. About 8 s/step on a T4, so under a
minute of compute for the 6-day forecast -- most wall-clock time is the checkpoint download and
the ARCO fetch.

In [ ]:
if RUN_MODELS:
    # Pangu's native lat grid, cut to Europe. Longitude stays global (the window
    # straddles the prime meridian) and is cropped after the run.
    lat_full = np.linspace(90.0, -90.0, 721)
    lat_keep = lat_full[(lat_full >= LAT_RANGE[0]) & (lat_full <= LAT_RANGE[1])]

    out_coords = OrderedDict({"variable": np.array(VARS), "lat": lat_keep})

    io = XarrayBackend()
    t0 = time.perf_counter()

    # TODO: run the autoregressive rollout with earth2studio.
    run.deterministic(...)  # <- replace ... with the arguments

    print(f"{NSTEPS} daily steps: {time.perf_counter() - t0:.0f} s")

In [ ]:
if RUN_MODELS:
    fcst = io.root.isel(time=0)
    fcst = (
        fcst
        .assign_coords(lon=(((fcst.lon + 180) % 360) - 180))
        .sortby("lon")
        .sel(lon=slice(*LON_RANGE))
    )
    save_output("pangu_forecast", fcst)
else:
    fcst = load_output("pangu_forecast")

print(f"{fcst.nbytes / 1e6:.0f} MB in memory")
fcst

### Watching the storm build, day by day

Six panels, one per forecast day: `t2m` shaded, `msl` contoured, 10 m wind as arrows. The model's
entire view of the event, in sequence.

In [ ]:
fig, axes = make_axes(2, 3, 16)
zmin, zmax = np.percentile(fcst.z500.values / 9.81, [2, 98])

for ax, t in zip(axes, VALID[1:]):
    day = fcst.sel(lead_time=t - INIT)
    m = ax.pcolormesh(
        day.lon,
        day.lat,
        day.z500 / 9.81,
        vmin=zmin,
        vmax=zmax,
        cmap="Spectral_r",
        shading="auto",
        **GEO,
    )
    cs = ax.contour(
        day.lon,
        day.lat,
        day.msl / 100,
        levels=np.arange(960, 1044, 4),
        colors="k",
        linewidths=0.6,
        **GEO,
    )
    ax.clabel(cs, cs.levels[::2], fontsize=6, fmt="%d")
    draw_wind(ax, day)
    ax.set_title(f"+{int((t - INIT) / np.timedelta64(1, 'h'))} h  ({str(t)[:10]})", fontsize=10)

fig.colorbar(m, ax=axes, shrink=0.7, label="500 hPa geopotential height [m]")
fig.suptitle(f"Pangu-Weather forecast initialised {str(INIT)[:13].replace('T', ' ')} UTC")
plt.show()

Shading is 500 hPa geopotential height, contours are `msl`, arrows are 10 m wind. Watch the
500 hPa field for a closed low **detaching** from the westerlies and stalling over Central Europe
instead of tracking east with the jet -- that cut-off, with the surface circulation wrapped tight
around it, is what pins the rain in place for days. Position/timing errors of
about a day, or a few hundred km, are normal for a single deterministic run at this range; day 6
in a blocked pattern is genuinely hard.

### Did it get the storm?

The day-6 forecast against what actually happened, side by side.

In [ ]:
T_CHECK = VALID[-1]
truth_check = cached(
    f"era5_truth_check_{RUN_TAG}",
    lambda: to_dataset(era5(np.array([T_CHECK]), ["msl", "z500", "u10m", "v10m"])),
).isel(time=0)
fc_check = fcst.isel(lead_time=-1)  # same instant as T_CHECK

fig, axes = make_axes(1, 2, 11)
both = np.concatenate([truth_check.z500.values.ravel(), fc_check.z500.values.ravel()]) / 9.81
zmin, zmax = np.percentile(both, [2, 98])

for ax, (title, ds) in zip(
    axes, [("ERA5 truth", truth_check), (f"Pangu24, +{LEAD_HOURS} h", fc_check)]
):
    m = ax.pcolormesh(
        ds.lon,
        ds.lat,
        ds.z500 / 9.81,
        vmin=zmin,
        vmax=zmax,
        cmap="Spectral_r",
        shading="auto",
        **GEO,
    )
    cs = ax.contour(
        ds.lon,
        ds.lat,
        ds.msl / 100,
        levels=np.arange(960, 1044, 4),
        colors="k",
        linewidths=0.6,
        **GEO,
    )
    ax.clabel(cs, cs.levels[::2], fontsize=6, fmt="%d")
    draw_wind(ax, ds)
    ax.set_title(title, fontsize=10)

fig.colorbar(m, ax=axes, shrink=0.8, label="500 hPa geopotential height [m]")
fig.suptitle(f"Valid {str(T_CHECK)[:13].replace('T', ' ')} UTC")
plt.show()

---
## 3. Verification: is it any good?

A plausible-looking map tells you little. What matters is whether the forecast beats the
cheap alternatives, so we score against two baselines:

- **Persistence**: nothing changes from the initial condition. Hard to beat at short lead times,
  trivial at long ones.
- **Climatology**: the 1990-2019 average for this day-of-year and hour. Impossible to beat at long
  lead times, because that is where every forecast converges. RMSE reaching the climatological
  RMSE means no information left.

A useful forecast lives between the two curves; where it crosses climatology is roughly the
predictability limit for this case.

Pangu's step is 24 h, so we verify at each of the six daily steps -- coarse, but that is the
model's actual temporal resolution, not a shortcut.

In [ ]:
vtimes = VALID
lead_h = fcst.lead_time / np.timedelta64(1, "h")
fc_v = fcst


def align_to(ds, ref):
    """Put ds on ref's exact lat/lon/lead_time labels, failing loudly on a shape mismatch."""
    assert (ds.sizes["lat"], ds.sizes["lon"]) == (ref.sizes["lat"], ref.sizes["lon"]), (
        f"grid mismatch: {dict(ds.sizes)} vs {dict(ref.sizes)}"
    )
    return ds.rename(time="lead_time").assign_coords(
        lat=ref.lat, lon=ref.lon, lead_time=ref.lead_time
    )


truth_v = align_to(cached(f"era5_truth_{RUN_TAG}", lambda: to_dataset(era5(vtimes, VARS))), fc_v)
clim_v = align_to(cached(f"wb2_clim_{RUN_TAG}", lambda: to_dataset(climatology(vtimes, VARS))), fc_v)

print("verification times:", len(vtimes))

In [ ]:
# Latitude weighting: a grid cell at 60N covers half the area of one at the equator.
W = np.cos(np.deg2rad(fc_v.lat))


def wrmse(a, b):
    # TODO: latitude-weighted RMSE of a against b, reduced over ("lat", "lon").
    ...


def wacc(f, o, c):
    """Anomaly correlation coefficient of forecast f against truth o, relative to climatology c."""
    fa, oa = f - c, o - c
    num = (fa * oa).weighted(W).mean(("lat", "lon"))
    den = np.sqrt(
        (fa**2).weighted(W).mean(("lat", "lon")) * (oa**2).weighted(W).mean(("lat", "lon"))
    )
    return num / den


# TODO: the persistence baseline is the truth at lead_time 0, held constant for every lead time.
persistence = ...

scores = {
    "Pangu24": wrmse(fc_v, truth_v),
    "persistence": wrmse(persistence, truth_v),
    "climatology": wrmse(clim_v, truth_v),
}

In [ ]:
SHOW = [
    ("z500", 1 / 9.81, "500 hPa geopotential height [m]"),
    ("msl", 1 / 100, "mean sea level pressure [hPa]"),
    ("q700", 1e3, "700 hPa specific humidity [g kg$^{-1}$]"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
styles = {
    "Pangu24": dict(color="C0", lw=2.0, marker="o", ms=4),
    "persistence": dict(color="0.45", lw=1.4, marker="o", ms=3),
    "climatology": dict(color="C3", lw=1.4, ls=":", marker="o", ms=3),
}

for ax, (var, scale, label) in zip(axes, SHOW):
    for name, sc in scores.items():
        ax.plot(lead_h, sc[var] * scale, label=name, **styles[name])
    ax.set_xlabel("lead time [h]")
    ax.set_title(label, fontsize=10)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("RMSE")
axes[0].legend(fontsize=8)
fig.suptitle(
    f"Latitude-weighted RMSE over Central Europe, initialised {str(INIT)[:13]} UTC", y=1.02
)
plt.tight_layout()
plt.show()

- **Day 1**: persistence is competitive -- the atmosphere has not changed much. Pangu should be
  ahead, but not dramatically.
- **Middle days**: the gap opens up -- where a forecast earns its keep.
- **Long lead**: the forecast curve bends towards climatology. Where its RMSE meets the
  climatological RMSE, there is no information left at that lead time.

Then the anomaly correlation (ACC), the score operational centres quote: did you predict the right
*departure* from normal? The useful-synoptic threshold is **ACC = 0.6**.

In [ ]:
acc = wacc(fc_v, truth_v, clim_v)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(lead_h, acc.z500, "C0", lw=2.0, marker="o", ms=4, label="Pangu24")
ax.axhline(0.6, color="C3", ls=":", label="useful-forecast threshold")
ax.set_xlabel("lead time [h]")
ax.set_ylabel("ACC")
ax.set_title("500 hPa geopotential anomaly correlation", fontsize=10)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.show()

**A caveat to insist on.** One storm, one initialisation, one deterministic run. RMSE and ACC from
six points are illustrative, not statistically robust -- a real evaluation scores tens to hundreds
of initialisation dates. Rankings from a single case are how bad papers get written.

No spread-skill ratio here: Pangu gives one forecast, not an ensemble. That diagnostic returns in
section 4, where CorrDiff's diffusion sampler genuinely does produce multiple plausible outcomes.

---
## 4. Downscaling to 2.2 km, hourly, with CorrDiff

Pangu never predicted any rain, and could not. It carries moisture, but no precipitation and no clouds -- and a 0.25 degree cell is about 28 km,
whereas the Alpine valleys and ridges that decide where 400 mm lands are 5-20 km apart. The
global model gives you the synoptic setup, not the impact.

**CorrDiff** closes both gaps. It is a *corrector diffusion* model in two stages:

1. a **regression** network predicting the conditional mean of the high-resolution field given the
   coarse one -- everything the terrain determines deterministically, and
2. a **diffusion** network sampling the residual -- small-scale structure that is plausible but
   not uniquely determined by the coarse input.

`CorrDiffCosmoEra5` maps ERA5 onto **COSMO-REA** (the DWD regional reanalysis over Europe) at 6 km
(`rea6`) or 2.2 km (`rea2`), and crucially outputs `TOT_PRECIP`.

We run it at its finest in both dimensions: **2.2 km**, and since ARCO-ERA5 is native hourly,
**hourly**. Then we switch on the diffusion half and look at a small ensemble of precipitation
fields.

First, free the GPU. `onnxruntime`'s memory arena sits outside PyTorch's allocator, so
`torch.cuda.empty_cache()` alone will not release it -- dropping the model reference does.

In [ ]:
if RUN_MODELS:
    model = None
    gc.collect()
    torch.cuda.empty_cache()

    if DEVICE == "cuda":
        free, total = torch.cuda.mem_get_info()
        print(f"VRAM free: {free / 1e9:.1f} GB / {total / 1e9:.1f} GB")

CorrDiff's backbone is a diffusion transformer with **neighbourhood attention**: each token
attends only to a local window, which keeps attention affordable at 2 km. That needs `natten`,
which ships compiled CUDA kernels, so the wheel must match the exact torch/CUDA/Python ABI.

Now load the model and restrict it to our domain.

`set_domain` is what makes this T4-friendly. The network is a DiT with rotary position embeddings,
so it is **crop-size agnostic**: trained at a fixed 2.2 km resolution, it runs on any sub-block
without retraining. Cropping to the Alpine domain instead of all of Europe cuts memory by a large
factor.

We load two copies, one per `mode`: `"mean"` (the fast single-pass regression, used for the
comparison and the hourly sequence) and `"diffusion"` (the ensemble, later). Same downloaded
package, so the second load is cheap.

In [ ]:
if RUN_MODELS:
    from earth2studio.models.dx import CorrDiffCosmoEra5

    t0 = time.perf_counter()
    cd_package = CorrDiffCosmoEra5.load_default_package()
    downscaler = CorrDiffCosmoEra5.load_model(
        cd_package, device=DEVICE, mode="mean", resolution=DOWNSCALE_RES
    )
    # TODO: crop the global downscaler to our box so it fits on a T4.
    alps = downscaler.set_domain(...).to(DEVICE)
    
    print(f"loaded in {time.perf_counter() - t0:.0f} s")

    ic = alps.input_coords()
    oc = alps.output_coords(ic)
    print(
        f"ERA5 input : {len(ic['lat'])} x {len(ic['lon'])} cells, {len(ic['variable'])} variables"
    )
    print(f"COSMO output: {np.asarray(oc['lat']).shape} cells at ~2.2 km")
else:
    print("no GPU -- CorrDiff outputs are read from cached_outputs/")

Note the asymmetry: a few thousand coarse input cells become a few hundred thousand output cells.
The model is not interpolating -- it generates structure consistent with the terrain and with how
COSMO-REA looks in this synoptic situation.

Feed it the ERA5 analysis at the peak of the event, and compare against the 0.25 degree input.

In [ ]:
if RUN_MODELS:
    x, coords = fetch_data(
        source=arco,
        time=np.array([DOWNSCALE_TIME], dtype="datetime64[ns]"),
        variable=ic["variable"],
        device=DEVICE,
    )

    # fetch_data returns (time, lead_time, variable, lat, lon); drop the singleton lead_time.
    x = x[:, 0]
    coords = OrderedDict((k, v) for k, v in coords.items() if k != "lead_time")
    # TODO: reorder/orient the fetched ERA5 tensor to the model's expected input coords `ic`.
    x, coords = map_coords(...)

    t0 = time.perf_counter()
    with torch.inference_mode():
        hires, hires_coords = alps(x, coords)
    print(f"downscaled in {time.perf_counter() - t0:.1f} s -> {tuple(hires.shape)}")

    # alps() adds then strips a batch axis (no "batch" key in coords), so this is
    # (sample, time, variable, H, W) with sample and time both 1.
    field = hires[0, 0].float().cpu().numpy()  # (variable, H, W)
    out_names = [str(v) for v in hires_coords["variable"]]
    lat2d = np.asarray(hires_coords["lat"])
    lon2d = np.asarray(hires_coords["lon"])

    era5_names = [str(v) for v in ic["variable"]]
    era5_t2m = x[0, era5_names.index("t2m")].float().cpu().numpy()
    era5_lat = np.asarray(ic["lat"])
    era5_lon = np.asarray(ic["lon"])

    save_output(
        "corrdiff_mean",
        xr.Dataset(
            {
                "field": (("variable", "y", "x"), field),
                "era5_t2m": (("era5_lat", "era5_lon"), era5_t2m),
            },
            coords={
                "variable": out_names,
                "lat2d": (("y", "x"), lat2d),
                "lon2d": (("y", "x"), lon2d),
                "era5_lat": era5_lat,
                "era5_lon": era5_lon,
            },
        ),
    )
else:
    _m = load_output("corrdiff_mean")
    field = _m["field"].values
    out_names = [str(v) for v in _m["variable"].values]
    lat2d = _m["lat2d"].values
    lon2d = _m["lon2d"].values
    era5_t2m = _m["era5_t2m"].values
    era5_lat = _m["era5_lat"].values
    era5_lon = _m["era5_lon"].values

print("available outputs:", out_names)

In [ ]:
def pick(names, *candidates):
    for c in candidates:
        if c in names:
            return names.index(c)
    raise KeyError(f"none of {candidates} in {names}")


# The COSMO downscaling window; every map in section 4 is framed to exactly this box.
extent = [
    DOWNSCALE_BBOX["lon_min"],
    DOWNSCALE_BBOX["lon_max"],
    DOWNSCALE_BBOX["lat_min"],
    DOWNSCALE_BBOX["lat_max"],
]

cosmo_t2m = field[pick(out_names, "t2m")]
tmin, tmax = np.percentile(cosmo_t2m, [1, 99])

fig, axes = make_axes(1, 2, 11, extent=extent, pad=1.3)

axes[0].pcolormesh(
    era5_lon, era5_lat, era5_t2m, vmin=tmin, vmax=tmax, cmap="RdYlBu_r", shading="auto", **GEO
)
axes[0].set_title("ERA5 2 m temperature, 0.25 deg (~28 km)", fontsize=10)

p1 = axes[1].pcolormesh(
    lon2d, lat2d, cosmo_t2m, vmin=tmin, vmax=tmax, cmap="RdYlBu_r", shading="auto", **GEO
)
axes[1].set_title(
    f"CorrDiff -> COSMO-{DOWNSCALE_RES.upper()} 2 m temperature (~2.2 km)", fontsize=10
)

if HAS_CARTOPY:  # pcolormesh on the coarse grid can nudge the view; pin it back
    for ax in axes:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

# Same tmin/tmax on both panels, so one shared colorbar covers both.
fig.colorbar(p1, ax=axes, location="bottom", shrink=0.6, label="K")
fig.suptitle(f"Valid {str(DOWNSCALE_TIME)[:13].replace('T', ' ')} UTC")
plt.show()

The right panel is the payoff: the Alps become a mountain range -- warm valley floors, cold
ridges -- carrying terrain that does not exist in the 0.25 degree input. The model learned that
relationship from four years of COSMO-REA.

### Precipitation, hour by hour

The field Pangu could not give us at all, at ARCO's native hourly cadence. The `N_HOURS`
consecutive hours ending at the peak, fetched and downscaled `HOURS_PER_BATCH` at a time so
peak memory does not grow with the sequence, then played back as an animation next to
ERA5's own precipitation on the same hours and the same colour scale.

Units: `TOT_PRECIP` comes back in metres (the `CosmoLexicon` scale), converted to mm below.

In [ ]:
if RUN_MODELS:
    # Inputs come from the cache (or are fetched once into it), then the model runs
    # in HOURS_PER_BATCH chunks so peak GPU memory is set by the chunk, not N_HOURS.
    xh_all, coords_all = cached_model_input(
        f"corrdiff_in_hourly_{DOWNSCALE_TAG}_{N_HOURS}h", DOWNSCALE_TIMES, ic
    )

    t0 = time.perf_counter()
    tp_chunks = []
    for i0 in range(0, len(DOWNSCALE_TIMES), HOURS_PER_BATCH):
        sl = slice(i0, i0 + HOURS_PER_BATCH)
        xh = xh_all[sl]
        coordsh = OrderedDict(coords_all)
        coordsh["time"] = coords_all["time"][sl]

        with torch.inference_mode():
            hires_h, hcoords_h = alps(xh, coordsh)

        names_h = [str(v) for v in hcoords_h["variable"]]
        # hires_h is (sample, time, variable, H, W) -- see the note two cells up.
        tp_chunks.append(hires_h[0, :, pick(names_h, "tp")].float().cpu().numpy() * 1e3)  # m -> mm

        del hires_h
        gc.collect()
        torch.cuda.empty_cache()
        print(f"  downscaled {min(i0 + HOURS_PER_BATCH, len(DOWNSCALE_TIMES))}/{len(DOWNSCALE_TIMES)} hours")

    tp_h = np.concatenate(tp_chunks)
    print(f"downscaled {len(DOWNSCALE_TIMES)} hours in {time.perf_counter() - t0:.1f} s -> {tp_h.shape}")

    save_output(
        "corrdiff_hourly",
        xr.Dataset(
            {"tp": (("time", "y", "x"), tp_h)},
            coords={
                "time": np.asarray(DOWNSCALE_TIMES),
                "lat2d": (("y", "x"), lat2d),
                "lon2d": (("y", "x"), lon2d),
            },
        ),
    )
else:
    _h = load_output("corrdiff_hourly")
    tp_h = _h["tp"].values
    DOWNSCALE_TIMES = _h["time"].values
    print(f"replayed {tp_h.shape[0]} cached hourly precip frames")

In [ ]:
# ERA5's own precipitation on the same hours. Goes through cached(), so it costs one
# live fetch ever and then ships with the repo.
era5_tp = cached(
    f"era5_tp_{DOWNSCALE_TAG}_{len(DOWNSCALE_TIMES)}h",
    lambda: to_dataset(era5(DOWNSCALE_TIMES, ["tp"]), subset=False).sel(
        lat=slice(DOWNSCALE_BBOX["lat_max"], DOWNSCALE_BBOX["lat_min"]),
        lon=slice(DOWNSCALE_BBOX["lon_min"], DOWNSCALE_BBOX["lon_max"]),
    ),
)
tp_e = era5_tp["tp"].values * 1e3  # m -> mm, the same units as TOT_PRECIP

# Side by side on one shared colour scale. CorrDiff adds terrain structure ERA5 cannot
# resolve, but note it does not add intensity: in mean mode it predicts a conditional
# mean, so its peaks are the *lower* of the two. Separate scales would hide both effects.
fig, axes = make_axes(1, 2, 12, extent=extent, pad=1.3)
vmax = max(1.0, float(np.percentile(np.concatenate([tp_e.ravel(), tp_h.ravel()]), 99.9)))

mesh_e = axes[0].pcolormesh(
    era5_tp.lon,
    era5_tp.lat,
    np.ma.masked_less(tp_e[0], 0.05),
    vmin=0,
    vmax=vmax,
    cmap="GnBu",
    shading="auto",
    **GEO,
)
mesh_c = axes[1].pcolormesh(
    lon2d,
    lat2d,
    np.ma.masked_less(tp_h[0], 0.05),
    vmin=0,
    vmax=vmax,
    cmap="GnBu",
    shading="auto",
    **GEO,
)
axes[0].set_title("ERA5, 0.25 deg (~28 km)", fontsize=10)
axes[1].set_title(f"CorrDiff -> COSMO-{DOWNSCALE_RES.upper()} (~2.2 km)", fontsize=10)

if HAS_CARTOPY:  # pcolormesh on the coarse grid can nudge the view; pin it back
    for ax in axes:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

fig.colorbar(mesh_c, ax=axes, location="bottom", shrink=0.5, label="hourly precipitation [mm]")
suptitle = fig.suptitle(" ")


def draw_frame(i):
    mesh_e.set_array(np.ma.masked_less(tp_e[i], 0.05))
    mesh_c.set_array(np.ma.masked_less(tp_h[i], 0.05))
    suptitle.set_text(str(DOWNSCALE_TIMES[i])[:13].replace("T", " ") + " UTC")
    return mesh_e, mesh_c


anim = animation.FuncAnimation(
    fig, draw_frame, frames=len(DOWNSCALE_TIMES), interval=400, blit=False
)
plt.close(fig)  # else the static first frame renders next to the player
HTML(anim.to_jshtml(default_mode="loop"))

The field a hydrologist actually wants: precipitation on windward slopes, evolving hour by hour,
with individual valleys resolved. Neither the rain nor this detail existed anywhere upstream of
this cell.

### How sure is the model? The diffusion sampler

Everything above used `mode="mean"` -- the regression network's single best guess: fast, but a
conditional *mean*, so smooth by construction and carrying no notion of its own uncertainty.

`mode="diffusion"` draws `number_of_samples` independent realisations from the learned
`p(y | x)`, each seeded separately -- a genuine ensemble, and far more expensive: 18 sampler steps
per sample, run at a single hour to fit the time budget. We draw `N_SAMPLES` (8 by default) and
show the `N_SHOW` wettest of them, since the sampler's first few members happen to be nearly
dry here; the spread (standard deviation) is taken across **all** of them.

At 2.2 km an 8-member ensemble is the slowest cell in the notebook, but it does fit a
free-tier Colab T4 -- the `set_domain` crop to `DOWNSCALE_BBOX` is what keeps it
affordable. Lower `N_SAMPLES` if you are short on time, or just replay the committed cache.

In [ ]:
if RUN_MODELS:
    t0 = time.perf_counter()
    downscaler_diff = CorrDiffCosmoEra5.load_model(
        cd_package, device=DEVICE, mode="diffusion", resolution=DOWNSCALE_RES
    )
    alps_diff = downscaler_diff.set_domain(**DOWNSCALE_BBOX).to(DEVICE)
    
    # TODO: set how many members the diffusion sampler draws.
    alps_diff.number_of_samples = ...  # <- replace
    alps_diff.seed = SEED

    print(f"loaded in {time.perf_counter() - t0:.0f} s, {N_SAMPLES} samples")

    xd, coordsd = cached_model_input(
        f"corrdiff_in_{DOWNSCALE_TAG}", [DOWNSCALE_TIME], ic
    )

    t0 = time.perf_counter()
    with torch.inference_mode():
        hires_d, hcoords_d = alps_diff(xd, coordsd)
    print(f"{N_SAMPLES} samples in {time.perf_counter() - t0:.0f} s -> {tuple(hires_d.shape)}")

    names_d = [str(v) for v in hcoords_d["variable"]]
    # hires_d is (sample, time, variable, H, W); keep the sample axis this time.
    tp_samples = hires_d[:, 0, pick(names_d, "tp")].float().cpu().numpy() * 1e3

    save_output(
        "corrdiff_diffusion",
        xr.Dataset(
            {"tp": (("sample", "y", "x"), tp_samples)},
            coords={"lat2d": (("y", "x"), lat2d), "lon2d": (("y", "x"), lon2d)},
        ),
    )
else:
    tp_samples = load_output("corrdiff_diffusion")["tp"].values
    N_SAMPLES = tp_samples.shape[0]  # match whatever the cached run produced
    print(f"replayed cached diffusion ensemble: {N_SAMPLES} samples")

In [ ]:
fig, axes = make_axes(1, N_SHOW + 1, 15, extent=extent, pad=1.3)
# The sampler returns members in its own order, and with this seed the first few are
# nearly dry -- a dull figure that undersells the ensemble. Show the N_SHOW wettest,
# labelled with their real member index so nothing is hidden. The spread panel below
# still uses every member.
wettest = np.argsort(tp_samples.sum(axis=(1, 2)))[::-1][:N_SHOW]
vmax = max(1.0, float(np.percentile(tp_samples[wettest], 99)))

for j, k in enumerate(wettest):
    m = axes[j].pcolormesh(
        lon2d,
        lat2d,
        np.ma.masked_less(tp_samples[k], 0.05),
        vmin=0,
        vmax=vmax,
        cmap="GnBu",
        shading="auto",
        **GEO,
    )
    axes[j].set_title(f"member {k}", fontsize=10)

# TODO: ensemble spread map = per-pixel standard deviation across ALL members.
spread = ...

smax = max(1e-3, float(np.percentile(spread, 98)))
ms = axes[-1].pcolormesh(
    lon2d, lat2d, spread, vmin=0, vmax=smax, cmap="magma", shading="auto", **GEO
)
axes[-1].set_title(f"spread (std across {tp_samples.shape[0]} members)", fontsize=10)

# Precip and spread are different quantities -> two colorbars, both along the
# bottom so constrained layout keeps the four panels the same size.
fig.colorbar(m, ax=list(axes[:N_SHOW]), location="bottom", shrink=0.8, label="TOT_PRECIP [mm]")
fig.colorbar(ms, ax=[axes[-1]], location="bottom", shrink=0.8, label="spread [mm]")

fig.suptitle(
    f"CorrDiff diffusion ensemble ({N_SHOW} wettest of {tp_samples.shape[0]} members), "
    f"{DOWNSCALE_RES.upper()}, {str(DOWNSCALE_TIME)[:13].replace('T', ' ')} UTC"
)
plt.show()

Look for two things. **Agreement on the large scale**: all samples should put the heaviest rain
over roughly the same slopes -- the part the coarse ERA5 input determines. (hard to verify on this small map) **Disagreement on fine
texture**: which exact ridge peaks, the shape of individual cells -- underdetermined by a 28 km
input, and the sampler shows its own uncertainty rather than papering over it with one smooth
guess. The spread map is where that disagreement concentrates; near zero everywhere means the
ensemble collapsed.

### Two caveats

**We downscaled the ERA5 analysis, not our own forecast.** The Pangu -> CorrDiff pipeline
(`run.diagnostic`) does not work out of the box: CorrDiff needs 47 ERA5 inputs including surface
pressure `sp` and the 100 m winds, which Pangu does not output. The fix is
`earth2studio.models.dx.DerivedSurfacePressure` for `sp` (the 100 m winds have no clean
substitute) -- exercise 4.

**A sharper field is not automatically a better one.** Regression mode under-predicts extremes;
diffusion mode looks realistic but any one sample is a draw, not the truth. Verifying downscaled
precipitation needs neighbourhood or distribution scores such as FSS or CRPS, not point-to-point
RMSE -- which would rank the blurry regression field higher for the wrong reason.

---
## 5. Exercises

1. **Move the initialisation.** Set `INIT` to `2024-09-11T00` and re-run section 2. Does the
   ACC = 0.6 crossing move? With only six daily points, how confident is that crossing?

2. **Chain the models.** Build the Pangu -> `DerivedSurfacePressure` -> CorrDiff pipeline and
   downscale the day-6 *forecast* rather than the analysis. Compare to section 4 to separate
   downscaling error from forecast error. (Precipitation not available, chose e.g. 'msl' or 't2m' instead.)

3. **Change the metric.** Compute a CRPS/FSS for the downscaled precipitation instead
   of RMSE, and show the regression-vs-diffusion ranking depends on which score you pick.

### Where to go next

- earth2studio: <https://nvidia.github.io/earth2studio>
- Pangu-Weather: <https://doi.org/10.1038/s41586-023-06185-3>
- CorrDiff: <https://arxiv.org/abs/2309.15214>
- WeatherBench 2: <https://sites.research.google/weatherbench/>
- ARCO-ERA5: <https://github.com/google-research/arco-era5>